In [ ]:
# 1. Instalar PySpark en la sesión de Colab
!pip install pyspark

# Importar librerías
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, avg, count, col
import time

# 2. Inicializar la SparkSession (Primer Objeto Spark)
spark = SparkSession.builder \
    .appName("Analisis_MINSA_Leishmaniosis") \
    .getOrCreate()

print("SparkSession iniciada correctamente.")

SparkSession iniciada correctamente.


In [ ]:
# Cargar el CSV indicando que el separador es punto y coma (sep=";")
df_atenciones = spark.read.csv(
    "Fact_Atenciones_Leish.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

# Mostrar el esquema corregido y los primeros registros
df_atenciones.printSchema()
df_atenciones.show(5)

# Operación de Estadística Descriptiva
print("--- Estadística Descriptiva de Cantidad_Casos ---")
df_atenciones.select("Cantidad_Casos").describe().show()

root
 |-- ID_Atencion: integer (nullable = true)
 |-- Codigo_Ubigeo: integer (nullable = true)
 |-- ID_Perfil: integer (nullable = true)
 |-- ID_Tiempo: integer (nullable = true)
 |-- Anio: integer (nullable = true)
 |-- Semana: integer (nullable = true)
 |-- Enfermedad: string (nullable = true)
 |-- Diagnostico: string (nullable = true)
 |-- Cantidad_Casos: integer (nullable = true)

+-----------+-------------+---------+---------+----+------+--------------------+-----------+--------------+
|ID_Atencion|Codigo_Ubigeo|ID_Perfil|ID_Tiempo|Anio|Semana|          Enfermedad|Diagnostico|Cantidad_Casos|
+-----------+-------------+---------+---------+----+------+--------------------+-----------+--------------+
|      45882|        80902|       50|      366|2007|     1|LEISHMANIASIS CUT...|      B55.1|             1|
|      45884|        80902|       17|      366|2007|     1|LEISHMANIASIS CUT...|      B55.1|             1|
|      45887|        80902|       17|      366|2007|     1|LEISHMANIASIS

In [ ]:
# Registrar la vista temporal para SparkSQL (Segundo Objeto Spark)
df_atenciones.createOrReplaceTempView("fact_atenciones")

# Medir tiempo de inicio
start_time = time.time()

# Operación de Agregación con SparkSQL
resultado_spark = spark.sql("""
    SELECT
        Anio,
        SUM(Cantidad_Casos) AS Total_Casos,
        ROUND(AVG(Cantidad_Casos), 2) AS Promedio_Casos
    FROM fact_atenciones
    GROUP BY Anio
    ORDER BY Anio ASC
""")

# Forzar la ejecución y mostrar los primeros resultados
resultado_spark.show(10)

end_time = time.time()
tiempo_ejecucion = end_time - start_time

print(f"Tiempo de ejecución en PySpark: {tiempo_ejecucion:.4f} segundos")

+----+-----------+--------------+
|Anio|Total_Casos|Promedio_Casos|
+----+-----------+--------------+
|2000|       5113|           1.0|
|2001|       4722|           1.0|
|2002|       6588|           1.0|
|2003|       6454|           1.0|
|2004|       6474|           1.0|
|2005|       7703|           1.0|
|2006|       7743|           1.0|
|2007|       9762|           1.0|
|2008|       7150|           1.0|
|2009|       6155|           1.0|
+----+-----------+--------------+
only showing top 10 rows
Tiempo de ejecución en PySpark: 2.2278 segundos
